In [70]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, classification_report, multilabel_confusion_matrix, accuracy_score
from sklearn.tree import DecisionTreeClassifier

Adding the area of the regions to the dataframe.

First, we'll create a dictionary with area codes and areas, then we iterate over a list of area codes from the accidents, create a new array based on them and add it to the dataframe.

In [71]:
accidents = pd.read_csv("koeti_riista_tau_001_en.csv", encoding='iso-8859-1')

regions = pd.read_csv('maakunnat_pinta_alat.csv')

areas = list(regions.iloc[:, 5])
area_codes = list(regions.iloc[:, 2])
areas_dict = dict(zip(area_codes, areas))

accident_areas = list(accidents.iloc[:, 8])

areas_arr = []

for accident_area in accident_areas:
    areas_arr.append(int(areas_dict[accident_area]))

accidents['regionArea'] = areas_arr

accidents

,id,eventTime,year,month,x,y,municipality,municipalityName,region,regionName,roadType,roadTypeShort,roadNumber,roadMaintenance,roadMaintenanceName,wildlifeSpecies,wildlifeSpeciesName,regionArea
0,1,2017-12-10T16:55:00.000+02:00,2017,12,278773.65900,6.980878e+06,743,Seinäjoki,14,Etelä-Pohjanmaa,3,Valtatie,18.0,1.0,Valtio,47507,Metsäkauris,13864807779
1,2,2017-09-03T13:00:00.000+03:00,2017,9,397501.00800,7.503084e+06,261,Kittilä,19,Lappi,4,Kantatie,80.0,1.0,Valtio,47503,Hirvi,94206757384
2,3,2017-09-18T07:00:00.000+03:00,2017,9,418353.84100,7.514105e+06,261,Kittilä,19,Lappi,6,Muu maantie,9552.0,1.0,Valtio,47503,Hirvi,94206757384
3,4,2017-12-01T17:00:00.000+02:00,2017,12,488970.82700,6.722504e+06,285,Kotka,8,Kymenlaakso,5,Seututie,357.0,1.0,Valtio,47629,Valkohäntäpeura,4622479273
4,5,2017-09-18T07:00:00.000+03:00,2017,9,418766.58000,7.506931e+06,261,Kittilä,19,Lappi,6,Muu maantie,9552.0,1.0,Valtio,47503,Hirvi,94206757384
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63937,63938,2021-10-04T09:30:00.000+03:00,2021,10,236501.85584,6.691606e+06,445,Parainen,2,Varsinais-Suomi,5,Seututie,180.0,1.0,Valtio,47507,Metsäkauris,10574665809
63938,63939,2021-09-20T19:51:48.017+03:00,2021,9,515750.42581,6.946907e+06,778,Suonenjoki,11,Pohjois-Savo,3,Valtatie,9.0,1.0,Valtio,47503,Hirvi,17589162923
63939,63940,2021-05-29T06:00:00.000+03:00,2021,5,245406.95714,6.712757e+06,202,Kaarina,2,Varsinais-Suomi,4,Kantatie,40.0,1.0,Valtio,47507,Metsäkauris,10574665809
63940,63941,2021-04-13T14:30:00.000+03:00,2021,4,238690.96803,6.705560e+06,853,Turku,2,Varsinais-Suomi,7,Puuttuu,NaN,99.0,Ei tietoa,47507,Metsäkauris,10574665809


Deriving time of day in seconds from eventTime.

First, we'll assign the values from the eventTime column into an array, then we'll iterate over the arrays strings and convert them to seconds, finally we append them to an array and add it to the dataframe.

In [72]:
seconds_arr = []

times = accidents.iloc[:, [1]].values

# '2017-12-10T16:55:00.000+02:00'

for time in times:
    times_of_day = time[0].split('T')[1]
    time_of_day = times_of_day.split('+')[0].split('.')[0].split(':')
    seconds = int(time_of_day[0]) * 3600 + int(time_of_day[1]) * 60 + int(time_of_day[2])
    seconds_arr.append(seconds)

accidents['timeOfDay'] = seconds_arr
accidents

,id,eventTime,year,month,x,y,municipality,municipalityName,region,regionName,roadType,roadTypeShort,roadNumber,roadMaintenance,roadMaintenanceName,wildlifeSpecies,wildlifeSpeciesName,regionArea,timeOfDay
0,1,2017-12-10T16:55:00.000+02:00,2017,12,278773.65900,6.980878e+06,743,Seinäjoki,14,Etelä-Pohjanmaa,3,Valtatie,18.0,1.0,Valtio,47507,Metsäkauris,13864807779,60900
1,2,2017-09-03T13:00:00.000+03:00,2017,9,397501.00800,7.503084e+06,261,Kittilä,19,Lappi,4,Kantatie,80.0,1.0,Valtio,47503,Hirvi,94206757384,46800
2,3,2017-09-18T07:00:00.000+03:00,2017,9,418353.84100,7.514105e+06,261,Kittilä,19,Lappi,6,Muu maantie,9552.0,1.0,Valtio,47503,Hirvi,94206757384,25200
3,4,2017-12-01T17:00:00.000+02:00,2017,12,488970.82700,6.722504e+06,285,Kotka,8,Kymenlaakso,5,Seututie,357.0,1.0,Valtio,47629,Valkohäntäpeura,4622479273,61200
4,5,2017-09-18T07:00:00.000+03:00,2017,9,418766.58000,7.506931e+06,261,Kittilä,19,Lappi,6,Muu maantie,9552.0,1.0,Valtio,47503,Hirvi,94206757384,25200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63937,63938,2021-10-04T09:30:00.000+03:00,2021,10,236501.85584,6.691606e+06,445,Parainen,2,Varsinais-Suomi,5,Seututie,180.0,1.0,Valtio,47507,Metsäkauris,10574665809,34200
63938,63939,2021-09-20T19:51:48.017+03:00,2021,9,515750.42581,6.946907e+06,778,Suonenjoki,11,Pohjois-Savo,3,Valtatie,9.0,1.0,Valtio,47503,Hirvi,17589162923,71508
63939,63940,2021-05-29T06:00:00.000+03:00,2021,5,245406.95714,6.712757e+06,202,Kaarina,2,Varsinais-Suomi,4,Kantatie,40.0,1.0,Valtio,47507,Metsäkauris,10574665809,21600
63940,63941,2021-04-13T14:30:00.000+03:00,2021,4,238690.96803,6.705560e+06,853,Turku,2,Varsinais-Suomi,7,Puuttuu,NaN,99.0,Ei tietoa,47507,Metsäkauris,10574665809,52200


Converting species names to one-hot values

With pandas get_dummies function, we'll first generate a dataframe with one-hot values for every species. Then, for the merge function to work, we add an id column to it and merge with accidents dataframe.

In [73]:
species_oh = pd.get_dummies(accidents['wildlifeSpeciesName'], dtype=int)

species_oh['id'] = range(len(species_oh))

accidents = accidents.merge(species_oh, on='id')
accidents

,id,eventTime,year,month,x,y,municipality,municipalityName,region,regionName,...,wildlifeSpecies,wildlifeSpeciesName,regionArea,timeOfDay,Hirvi,Kuusipeura,Metsäkauris,Metsäpeura,Valkohäntäpeura,Villisika
0,1,2017-12-10T16:55:00.000+02:00,2017,12,278773.65900,6.980878e+06,743,Seinäjoki,14,Etelä-Pohjanmaa,...,47507,Metsäkauris,13864807779,60900,1,0,0,0,0,0
1,2,2017-09-03T13:00:00.000+03:00,2017,9,397501.00800,7.503084e+06,261,Kittilä,19,Lappi,...,47503,Hirvi,94206757384,46800,1,0,0,0,0,0
2,3,2017-09-18T07:00:00.000+03:00,2017,9,418353.84100,7.514105e+06,261,Kittilä,19,Lappi,...,47503,Hirvi,94206757384,25200,0,0,0,0,1,0
3,4,2017-12-01T17:00:00.000+02:00,2017,12,488970.82700,6.722504e+06,285,Kotka,8,Kymenlaakso,...,47629,Valkohäntäpeura,4622479273,61200,1,0,0,0,0,0
4,5,2017-09-18T07:00:00.000+03:00,2017,9,418766.58000,7.506931e+06,261,Kittilä,19,Lappi,...,47503,Hirvi,94206757384,25200,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63936,63937,2021-06-06T08:30:00.000+03:00,2021,6,395363.59416,6.687913e+06,92,Vantaa,1,Uusimaa,...,47507,Metsäkauris,9109841425,30600,0,0,1,0,0,0
63937,63938,2021-10-04T09:30:00.000+03:00,2021,10,236501.85584,6.691606e+06,445,Parainen,2,Varsinais-Suomi,...,47507,Metsäkauris,10574665809,34200,1,0,0,0,0,0
63938,63939,2021-09-20T19:51:48.017+03:00,2021,9,515750.42581,6.946907e+06,778,Suonenjoki,11,Pohjois-Savo,...,47503,Hirvi,17589162923,71508,0,0,1,0,0,0
63939,63940,2021-05-29T06:00:00.000+03:00,2021,5,245406.95714,6.712757e+06,202,Kaarina,2,Varsinais-Suomi,...,47507,Metsäkauris,10574665809,21600,0,0,1,0,0,0


Next we'll drop unnecessary columns and then rows with null values.

In [74]:
print(len(accidents))

print(len(accidents.dropna()))

accidents = accidents.drop(columns=['eventTime', 'municipalityName', 'regionName', 'roadTypeShort', 'roadMaintenanceName', 'wildlifeSpecies', 'wildlifeSpeciesName', 'roadNumber', 'roadMaintenance'])

accidents = accidents.dropna()

print(len(accidents))

63941
54511
63937


Data sharing

We first separate feature and label columns from the dataframe, then split and fit them to a linear regression model.

In [75]:
features = ['year', 'month', 'municipality', 'region', 'roadType',  'timeOfDay', 'x', 'y']
labels = ['Hirvi', 'Kuusipeura', 'Metsäkauris', 'Metsäpeura', 'Valkohäntäpeura', 'Villisika']

X, y = accidents[features].values, accidents[labels].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.7, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)

r2 = r2_score(y_test, y_pred)

print(f"Mean squared error: {mse:.2f}")
print(f"R2 score: {r2:.2f}")


Mean squared error: 0.10
R2 score: 0.01


Model training

Next, we train a decision tree using the training data and create a prediction

In [76]:
clf = DecisionTreeClassifier()

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

Model evaluation

Finally, we evaluate the results using various evaluation functions from sklearn.metrics

In [77]:
print(classification_report(y_test, y_pred))

print(multilabel_confusion_matrix(y_test, y_pred))

print()
print(accuracy_score(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.19      0.20      0.20      6347
           1       0.00      0.00      0.00        60
           2       0.39      0.39      0.39     15790
           3       0.04      0.05      0.04       232
           4       0.54      0.52      0.53     22204
           5       0.02      0.02      0.02       123

   micro avg       0.43      0.43      0.43     44756
   macro avg       0.20      0.20      0.20     44756
weighted avg       0.43      0.43      0.43     44756
 samples avg       0.43      0.43      0.43     44756

[[[32750  5659]
  [ 5049  1298]]

 [[44631    65]
  [   60     0]]

 [[19352  9614]
  [ 9596  6194]]

 [[44269   255]
  [  221    11]]

 [[12549 10003]
  [10671 11533]]

 [[44511   122]
  [  121     2]]]

0.4253731343283582
